In [75]:
import duckdb 
import pandas as pd

# Exploration of admissions data

In [25]:
# Create a duckdb session/connection
admissions_con: duckdb.DuckDBPyConnection = duckdb.connect()

# Read SQL file
with open("sql_files/admissions.sql", "r") as f:
    sql: str = f.read()

admissions_result: pd.DataFrame = admissions_con.execute(sql).df() # remember to fetch result using df()
admissions_result

,avg(subject_count)
0,2.443603


# Exploration of diagnoses data

In [142]:
diagnoses_con: duckdb.DuckDBPyConnection = duckdb.connect()

# Read SQL file
with open("sql_files/diagnoses_sql/active_cancer.sql", "r") as f:
    sql: str = f.read()

diagnoses_result: pd.DataFrame = diagnoses_con.execute(sql).df() # remember to fetch result using df()
diagnoses_result

,count(DISTINCT subject_id)
0,29369


In [143]:
total_diagnoses_patients_con: duckdb.DuckDBPyConnection = duckdb.connect()

diagnoses_sql_queries = ["active_cancer", "personal_history_cancer"]

# Perform SQL queries so multiple files can be loaded. 
for query in diagnoses_sql_queries:
    with open(f"sql_files/diagnoses_sql/{query}.sql", "r") as f:
        sql: str = f.read()
        total_diagnoses_patients_con.execute(sql).df()
        
with open(f"sql_files/diagnoses_sql/history_and_active.sql", "r") as f:
    total_cancer_patients_sql: str = f.read()


total_diagnoses_result: pd.DataFrame = total_diagnoses_patients_con.execute(total_cancer_patients_sql).df()
total_diagnoses_result

,subject_id
0,13162037
1,13241979
2,13242005
3,13281088
4,13122104
...,...
48433,16540359
48434,19286193
48435,19322323
48436,19339534


# Exploration of prescriptions data

1. Look at how many patients have had antineoplasts or specific drugs across the entire dataset. 
2. Perform 1. but within the total number of cancer patients. 

In [141]:
prescriptions_con: duckdb.DuckDBPyConnection = duckdb.connect()

# Read SQL file
with open("sql_files/prescriptions_sql/prescriptions_count.sql", "r") as f:
    sql: str = f.read()

prescriptions_result: pd.DataFrame = prescriptions_con.execute(sql).df() # remember to fetch result using df()
prescriptions_result

,subject_id
0,10035618
1,10043050
2,10049746
3,10054464
4,10108214
...,...
3217,19860377
3218,19912102
3219,19940147
3220,19960665


# Find intersection of diagnoses and prescriptions 

In [113]:
common_diag_prescript_patients: pd.DataFrame = total_diagnoses_result[total_diagnoses_result["subject_id"].isin(prescriptions_result["subject_id"])]
common_diag_prescript_patients

,subject_id
8,10554780
31,10777749
34,10793648
92,15704389
119,11600572
...,...
48381,13992510
48398,18176556
48399,11134447
48425,11973164


# Find cancer-related procedures

In [114]:
procedures_con: duckdb.DuckDBPyConnection = duckdb.connect()

# Read SQL file
with open("sql_files/procedures_sql/procedures_count.sql", "r") as f:
    sql: str = f.read()

procedures_result: pd.DataFrame = procedures_con.execute(sql).df() # remember to fetch result using df()
procedures_result

,subject_id
0,10002155
1,10003019
2,10035168
3,10064049
4,10098008
...,...
2886,19930120
2887,19930554
2888,19949164
2889,19956654


# Find intersection of cancer diagnoses and procedures

In [115]:
common_diag_procedure_patients: pd.DataFrame = total_diagnoses_result[total_diagnoses_result["subject_id"].isin(procedures_result["subject_id"])]
common_diag_procedure_patients

,subject_id
8,10554780
13,10566950
20,10666715
34,10793648
45,10859137
...,...
48196,18529025
48220,19034797
48253,17332316
48399,11134447


# Get study counts for echocardiography

In [148]:
echo_study_con: duckdb.DuckDBPyConnection = duckdb.connect()

# Read SQL file
with open("sql_files/echo_sql/study_counts.sql", "r") as f:
    sql: str = f.read()

echo_study_result: pd.DataFrame = echo_study_con.execute(sql).df() # remember to fetch result using df()
echo_study_result

,subject_id
0,10020306
1,10079700
2,10080695
3,10141438
4,10156119
...,...
4574,19895035
4575,19895419
4576,19897314
4577,19898439


In [149]:
common_diag_echo_patients: pd.DataFrame = total_diagnoses_result[total_diagnoses_result["subject_id"].isin(echo_study_result["subject_id"])]
common_diag_echo_patients

,subject_id
16,13467793
56,16659583
69,16955674
124,12378259
131,12476400
...,...
48213,15846185
48231,19213022
48392,15743458
48400,18871003


# Look for dangerous LVEF counts in echocardiography dataset

In [144]:
LVEF_study_con: duckdb.DuckDBPyConnection = duckdb.connect()

# Read SQL file
with open("sql_files/echo_sql/LVEF_counts.sql", "r") as f:
    sql: str = f.read()

LVEF_study_result: pd.DataFrame = LVEF_study_con.execute(sql).df() # remember to fetch result using df()
LVEF_study_result

,subject_id
0,10000980
1,10002430
2,10013310
3,10016150
4,10017492
...,...
15153,19979360
15154,19994379
15155,19995780
15156,19997293


In [145]:
common_diag_LVEF_patients: pd.DataFrame = total_diagnoses_result[total_diagnoses_result["subject_id"].isin(LVEF_study_result["subject_id"])]
common_diag_LVEF_patients

,subject_id
4,13122104
5,13140505
13,13274532
16,13467793
48,16665574
...,...
48386,11862609
48403,19154843
48405,19260901
48418,13221453
